# Run at start

In [ ]:
# imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    PowerTransformer, StandardScaler, OneHotEncoder, FunctionTransformer
    )
from sklearn.compose import ColumnTransformer
from pandas.api.types import is_numeric_dtype
import joblib

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

In [ ]:
dumps = '../Dumps/'
service_cols = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies"
]
target = 'Churn'
random_state = 42

In [ ]:
# loading dataset, display 10 samples of it
df = pd.read_csv('../Data/WA_Fn-UseC_-Telco-Customer-Churn.csv').drop(columns=['customerID'])
df.sample(10, random_state=random_state)

In [ ]:
# df['SeniorCitizen'] = df['SeniorCitizen'].astype(str)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

# Helper Functions

In [ ]:
def get_binary_categorical_columns() -> list:
    return [
        col for col in df.columns
        if not is_numeric_dtype(df[col]) and df[col].nunique(dropna=True) == 2
    ]


def get_multi_categorical_columns(df: pd.DataFrame) -> list:
    return [
        col for col in df.columns
        if not is_numeric_dtype(df[col]) and df[col].nunique(dropna=True) > 2
    ]


def get_numeric_columns(df: pd.DataFrame) -> list:
    return [col for col in df.columns if is_numeric_dtype(df[col]) and df[col].nunique(dropna=True) > 2]

In [ ]:
def group_values(feature_name, bins_num, X_train, df_save):
    min_value = X_train[feature_name].min()
    max_value = X_train[feature_name].max()

    bins = np.linspace(
        min_value,
        max_value,
        bins_num + 1
    )

    labels = [
        f"{bins[i]:.0f}-{bins[i+1]:.0f}"
        for i in range(len(bins) - 1)
    ]

    print("Labels for the ranges:", labels)

    name = f"{feature_name}Group"

    df_save[name] = pd.cut(
        df_save[feature_name],
        bins=bins,
        labels=labels,
        include_lowest=True
    )

    return name

In [ ]:
def get_missing_pressent():
    missing_percent = df.isnull().mean() * 100
    missing = df.isnull().sum()

    missing_table = pd.DataFrame({
        "Missing Count": missing,
        "Missing %": missing_percent
    })

    print(missing_table.sort_values("Missing %", ascending=False))

# Drop Duplicates

In [ ]:
df.drop_duplicates(inplace=True)
print("Number of Duplicate Rows:", df.duplicated().sum())

# Missing Values

In [ ]:
get_missing_pressent()

In [ ]:
missing_value_rows = df[df['TotalCharges'].isna()]
missing_value_rows[['tenure','MonthlyCharges', 'TotalCharges', 'Churn']]

In [ ]:
missing_value_rows[['tenure', 'MonthlyCharges']].describe()

In [ ]:
compare_df = df[["MonthlyCharges", "tenure", "TotalCharges"]].copy()
compare_df["ExpectedTotalCharges"] = (compare_df["MonthlyCharges"] * dcompare_dff["tenure"])
compare_df["ChargeDifference"] = (compare_df["TotalCharges"] - compare_df["ExpectedTotalCharges"])

compare_df.sample(10, random_state=random_state)

In [ ]:
avrg_offset = compare_df["ChargeDifference"].mean()
print(f"The Mean of the Offset: {avrg_offset}")

In [ ]:
temp = "No Row with 0 Total Charges" if df[df['TotalCharges'] == 0 ].empty else 'exits'
print(temp)

In [ ]:
df['TotalCharges'] = df['TotalCharges'].fillna(0)
get_missing_pressent()

# Feature Extraction & Feature Aggregation (For all of the dataset)

In [ ]:
# Services
df["ServiceCount"] = sum(
    (df[col] == "Yes").astype(int)
    for col in service_cols
)

df["ServiceCount"]

In [ ]:
# Streaming
df["StreamingCount"] = (
    (df["StreamingTV"] == "Yes").astype(int)
    + (df["StreamingMovies"] == "Yes").astype(int)
)

df["StreamingCount"]

In [ ]:
# Security/support
df["SecuritySupportCount"] = (
    (df["OnlineSecurity"] == "Yes").astype(int)
    + (df["TechSupport"] == "Yes").astype(int)
    + (df["DeviceProtection"] == "Yes").astype(int)
)

df["SecuritySupportCount"]

In [ ]:
# Contract
df["IsMonthToMonth"] = (
    df["Contract"] == "Month-to-month"
).astype(int)

df["IsMonthToMonth"]

In [ ]:
# Payment
df["IsElectronicCheck"] = (
    df["PaymentMethod"] == "Electronic check"
).astype(int)

df["IsElectronicCheck"]

In [ ]:
# Household
df["HasFamily"] = (
    (df["Partner"] == "Yes") |
    (df["Dependents"] == "Yes")
).astype(int)

df["HasFamily"]

In [ ]:
# Tenure
df["IsNewCustomer"] = (df["tenure"] <= 6).astype(int)
df[["IsNewCustomer"]]

In [ ]:
# Spending
df["AverageMonthlySpend"] = (df["TotalCharges"] / (df["tenure"] + 1))

# df["ExpectedTotalCharges"] = (df["MonthlyCharges"] * df["tenure"])

# df["ChargeDifference"] = (df["TotalCharges"] - df["ExpectedTotalCharges"])

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.sample(5)

# Make the Splits

In [ ]:
# Making the splits
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=random_state,
    stratify=y
)
print(f"Trainset Length: Train: {len(X_train)}, Test: {len(y_train)}")
print(f"Testset Length: Train: {len(X_train)}, Test: {len(y_train)}")

# Feature Extraction & Feature Aggregation (For Training set only)

In [ ]:
# Spending
median_mon_charges = X_train["MonthlyCharges"].median()

X_train["NewCustomer_HighCharge"] = ((X_train["tenure"] <= 6) &
 (X_train["MonthlyCharges"] > median_mon_charges)).astype(int)

X_test["NewCustomer_HighCharge"] = ((X_test["tenure"] <= 6) &
 (X_test["MonthlyCharges"] > median_mon_charges)).astype(int)

train1_name = group_values('MonthlyCharges', 5,X_train, X_train)
train2_name = group_values('TotalCharges',5, X_train, X_train)

group_values('MonthlyCharges', 5, X_train, X_test)
group_values('TotalCharges',5, X_train, X_test)

X_train[["AverageMonthlySpend", "ExpectedTotalCharges", "ChargeDifference", "NewCustomer_HighCharge" , 
train1_name, train2_name]]

In [ ]:
# Tenure
df["IsOldCustomer"] = (df["tenure"] > df['tenure'].median()).astype(int)

# --------------------------------------------------------------------
train_name = group_values('tenure',5,X_train, X_train)
test_name = group_values('tenure',5,X_train, X_test)

X_train[["IsOldCustomer", group_name]]

In [ ]:
print(df.shape)
df.info()

In [ ]:
df.sample(5)

In [ ]:
# saving the splits
df_train = pd.concat([X_train, y_train], axis=1)
df_test = pd.concat([X_test, y_test], axis=1)

df_train['split'] = 0
df_test['split'] = 1

pro_df = pd.concat([df_train, df_test], axis=0, ignore_index=True)
pro_df.to_parquet(f'{dumps}pro_df.parquet')

pro_df

# Making The Pipline (~~Balance~~, Prepare Splits, Scalers, Skew balancer)

*\**NEW*\* How Values are to be encoded:*
- *Binary Encoding* ('Yes': 1, 'No': 0, 'Male': 1, 'Female': 0): 'gender', 'Partner', 'Dependents', 'PhoneService', 'PaperlessBilling', 'Churn'

- *One-Hot encoding*: 'PaymentMethod', 'InternetService', 'Contract', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'

*> For the imbalance isues with the target:* I'll try weighted class first

In [ ]:
manual_cols = get_binary_categorical_columns()
onehot_cols = get_multi_categorical_columns()
numeric_cols = get_numeric_columns()

X = df.drop(columns=target)
y = df[target]

print("Numeric Cols: ", numeric_cols)
print("Manual Cols: ", manual_cols)
print("Onehot Cols: ", onehot_cols)

*Creating the Processing Pipelines*

In [ ]:
# Numeric Pipeline: SimpleImputer(median) -> yeo_johnson -> StandardScaler
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("yeo_johnson", PowerTransformer(method="yeo-johnson")),
    ("scaler", StandardScaler())
])

In [ ]:
# Manual Pipeline: SimpleImputer(most_frequent) -> OneHotEncoder
def custom_mapper(X):
    mapping = {
        'Yes': 1, 'No': 0,
        'Male': 1, 'Female': 0,
    }
    return X.map(mapping)

# 2. Build the working pipeline
manual_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", FunctionTransformer(custom_mapper))
])

In [ ]:
# Onehot Pipeline: SimpleImputer(most_frequent) -> OneHotEncoder
onehot_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

In [ ]:
# Ensemble the full preprocessor
preprocessor = ColumnTransformer([
    (
        "numeric",
        numeric_pipeline,
        numeric_cols
    ),
    (
        "manual",
        manual_pipeline,
        manual_cols
    ),
        (
        "onehot",
        onehot_pipeline,
        onehot_cols
    )
])

joblib.dump(preprocessor, f'{dumps}preprocessor.pkl')

*How to make the full pipeline*

In [ ]:
# full_pipeline = Pipeline([
#     ("preprocessor", preprocessor),
#     ("model", model)
# ])

*Making the splits*

In [ ]:
skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=random_state
)

joblib.dump(skf, f'{dumps}skf.pickle')

In [ ]:
# results = cross_validate(
#     full_pipeline,
#     X_train,
#     y_train,
#     cv=skf,
#     scoring=[
#         "roc_auc",
#         "precision",
#         "recall",
#         "f1"
#     ]
# )